In [ ]:
!pip install anthropic --quiet

import anthropic
import os

# Wpisz swój klucz API:
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-WPISZ_TUTAJ"

client = anthropic.Anthropic()

SYSTEM = """
Jesteś Super AI — mądry, pomocny i zawsze piszesz po polsku.
Rozumiesz polskie słownictwo, slang i skróty.
Odpowiadaj naturalnie, jak dobry znajomy.
"""

NASTROJE = {"pozytywny": "😊", "negatywny": "😟", "neutralny": "😐",
            "pytający": "🤔", "zaskoczony": "😲", "żartobliwy": "😄"}

historia = []

def nastoj(tekst):
    r = client.messages.create(
        model="claude-opus-4-8", max_tokens=10,
        system="Odpowiedz jednym słowem po polsku: pozytywny, negatywny, neutralny, pytający, zaskoczony lub żartobliwy.",
        messages=[{"role": "user", "content": tekst}]
    )
    n = r.content[0].text.strip().lower()
    return f"{NASTROJE.get(n, '💬')} {n}"

print("🤖 Super AI gotowy! Wpisz wiadomość (lub 'koniec' żeby zakończyć)\n")

while True:
    wiad = input("👤 Ty: ").strip()
    if wiad.lower() in ("koniec", "exit", "quit"):
        print("🤖 Do zobaczenia!")
        break
    if not wiad:
        continue

    print(f"🎭 Nastrój: {nastoj(wiad)}")
    historia.append({"role": "user", "content": wiad})

    print("🤖 AI: ", end="", flush=True)
    odpowiedz = ""
    with client.messages.stream(
        model="claude-opus-4-8", max_tokens=1024,
        system=SYSTEM, thinking={"type": "adaptive"},
        messages=historia
    ) as s:
        for fragment in s.text_stream:
            print(fragment, end="", flush=True)
            odpowiedz += fragment

    print("\n")
    historia.append({"role": "assistant", "content": odpowiedz})